# Weights & Watts, episode 1, on Colab

This runs the episode's tests of **MiniMax H3-Image** and **P-H3-Image** (Pruna's rebuild of it) on a rented Colab card, with the same ComfyUI, the same 4-bit model files and the same settings as the episode. Every take adds a row to a CSV: the seconds, the card's memory at its highest, the system's memory at its highest, the card's average and highest watts, its watt-hours, and the compute units the take used.

**Before you start:** choose *Runtime → Change runtime type → L4* or *A100*. Colab then shows what the runtime costs an hour in the *Resources* panel. Colab doesn't hand that number to the code, so type it in below if you want units in the CSV.

What it fetches: the model files are 29.4 GB for MiniMax H3-Image and 26.8 GB for P-H3-Image. The models come under their makers' own licences, linked in the repository's README.

Code by Kushal Sharma, MIT License. https://github.com/kushalsharma/weights-and-watts

In [ ]:
# Settings. Change these, then run every cell in order (Runtime → Run all).
MODEL = "h3-image"              # "h3-image" (MiniMax) or "p-h3-image" (Pruna)
PROMPTS = ["c-size-4mp"]        # names from the list printed below; the size test needs no pictures
TAKES = 2                       # the episode made 2: the first on a fixed seed, the second on a new one
UNITS_PER_HOUR = None           # from Colab's Resources panel, e.g. 1.94; None leaves the units column empty
DOLLARS_PER_UNIT = None         # what a compute unit cost you, e.g. 0.10; None leaves the cost column empty
PICTURES = {}                   # for prompts that bring her in, e.g. {"face": "/content/face.png", "body": "/content/body.png"}

REPO = "https://github.com/kushalsharma/weights-and-watts"

In [ ]:
import json, os, shlex, subprocess, sys, time
STARTED = time.time()

def sh(cmd):
    """Runs a shell command and shows its output as it comes."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
sh("free -g | head -2; df -h /content | tail -1")

## Get the code and ComfyUI

ComfyUI is checked out at the commit the episode used. Colab's own PyTorch is kept, as it was for the episode's Colab runs.

In [ ]:
if not os.path.exists("/content/weights-and-watts"):
    sh(f"git clone -q {REPO} /content/weights-and-watts")
EP = "/content/weights-and-watts/ep01-h3-image"
COMMIT = json.load(open(f"{EP}/models.json"))["comfyui"]["commit"]
if not os.path.exists("/content/ComfyUI/main.py"):
    sh(f"git init -q /content/ComfyUI && git -C /content/ComfyUI fetch -q --depth 1 https://github.com/Comfy-Org/ComfyUI {COMMIT} "
       f"&& git -C /content/ComfyUI -c advice.detachedHead=false checkout -q FETCH_HEAD")
sh("pip install -q -r /content/ComfyUI/requirements.txt huggingface_hub")
sh("python -c 'import torch; print(\"torch\", torch.__version__, \"CUDA\", torch.version.cuda, \"|\", torch.cuda.get_device_name(0))'")
sh(f"python {EP}/run_takes.py --list")

## Get the model files

Straight from the makers' Hugging Face pages, at the revisions the episode used, into ComfyUI's models folder.

In [ ]:
sh(f"python {EP}/get_models.py {MODEL} --to /content/ComfyUI/models")

## Run the takes

ComfyUI starts for the run and stops after it, so the first take includes loading the model, as a first take did in the episode. Pictures and a JSON record of each take go to `/content/runs`.

In [ ]:
args = ["python", f"{EP}/run_takes.py", "--comfy", "/content/ComfyUI", "--comfy-flags=--disable-auto-launch",
        "--model", MODEL, "--takes", str(TAKES), "--out", "/content/runs"]
for p in PROMPTS:
    args += ["--prompt", p]
for name, path in PICTURES.items():
    args += ["--picture", f"{name}={path}"]
if UNITS_PER_HOUR:
    args += ["--units-per-hour", str(UNITS_PER_HOUR)]
if DOLLARS_PER_UNIT:
    args += ["--dollars-per-unit", str(DOLLARS_PER_UNIT)]
sh(shlex.join(args))

## Your rows, beside ours

Ours come from `results/our-takes.csv` in the repository: the same prompt, the same model, on the RTX 3060 at home and on Colab.

In [ ]:
import pandas as pd
cols = ["prompt", "take", "model", "card", "width", "height", "seed", "seconds", "card_gb_peak", "system_gb_peak",
        "watts_avg", "watts_peak", "watt_hours", "colab_units"]
yours = pd.read_csv("/content/runs/takes.csv")
ours = pd.read_csv(f"{EP}/results/our-takes.csv")
names = {"h3-image": "MiniMax H3-Image", "p-h3-image": "P-H3-Image"}
print("Yours:")
display(yours[cols + ["error"]].tail(20))
print("Ours, same prompts and model:")
display(ours[(ours.prompt.isin(PROMPTS)) & (ours.model == names[MODEL])][cols + ["place"]])

In [ ]:
# The pictures from this run.
from IPython.display import Image, display
for pic in yours.picture.dropna().tail(6):
    print(pic)
    display(Image(f"/content/runs/{pic}", width=384))

## What this session used

Colab counts units for as long as the runtime is connected, start-up and downloads included. *Runtime → Disconnect and delete runtime* stops the count.

In [ ]:
hours = (time.time() - STARTED) / 3600
print(f"This notebook has been running for {hours * 60:.0f} minutes.")
if UNITS_PER_HOUR:
    print(f"At {UNITS_PER_HOUR} units an hour that is about {hours * UNITS_PER_HOUR:.2f} units, everything included.")